In [1]:
# ===============================
# 1️⃣ Import Libraries
# ===============================

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler


# ===============================
# 2️⃣ Load Dataset
# ===============================

df = pd.read_csv("assam_procurement_cleaned.csv")

print("Original Shape:", df.shape)

# ===============================
# 3️⃣ Rename Columns
# ===============================

df = df.rename(columns={
    'tender/value/amount': 'amount',
    'tender/tenderPeriod/durationInDays': 'duration_days',
    'tender/numberOfTenderers': 'num_bidders',
    'tender/procurementMethod': 'proc_method',
    'buyer/name': 'buyer',
    'fiscal_year': 'year',
    'tender/status': 'status'
})

# ===============================
# 4️⃣ Handle Missing Values SAFELY
# ===============================

# Convert numeric columns properly
numeric_cols = ['amount', 'duration_days', 'num_bidders']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical columns
categorical_cols = ['proc_method', 'buyer', 'status']

for col in categorical_cols:
    if df[col].isnull().all():
        df[col] = "Unknown"
    else:
        mode_val = df[col].mode()
        if len(mode_val) > 0:
            df[col] = df[col].fillna(mode_val[0])
        else:
            df[col] = df[col].fillna("Unknown")

print("After cleaning:", df.shape)

# ===============================
# 5️⃣ Feature Engineering
# ===============================

buyer_count = df['buyer'].value_counts()
df['buyer_frequency'] = df['buyer'].map(buyer_count)

df['proc_method_encoded'] = df['proc_method'].astype('category').cat.codes
df['status_encoded'] = df['status'].astype('category').cat.codes

# ===============================
# 6️⃣ Prepare ML Features
# ===============================

features = [
    'amount',
    'duration_days',
    'num_bidders',
    'buyer_frequency',
    'proc_method_encoded',
    'status_encoded'
]

X = df[features]

print("Feature Shape:", X.shape)

# ===============================
# 7️⃣ Scaling
# ===============================

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ===============================
# 8️⃣ Train Isolation Forest
# ===============================

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)

df['anomaly_score'] = model.decision_function(X_scaled)

# Risk Score (0–100)
risk_scaler = MinMaxScaler(feature_range=(0,100))
df['risk_score'] = risk_scaler.fit_transform(-df[['anomaly_score']])

# Risk Grade
def categorize(score):
    if score > 70:
        return "High"
    elif score > 40:
        return "Medium"
    else:
        return "Low"

df['risk_grade'] = df['risk_score'].apply(categorize)

# ===============================
# 9️⃣ Risk Reasons
# ===============================

reasons = []

for _, row in df.iterrows():
    r = []

    if row['num_bidders'] <= 1:
        r.append("Low bidder competition")

    if row['duration_days'] < 3:
        r.append("Very short tender duration")

    if row['amount'] > df['amount'].quantile(0.90):
        r.append("Unusually high contract value")

    if row['buyer_frequency'] > df['buyer_frequency'].quantile(0.90):
        r.append("High buyer concentration")

    if not r:
        r.append("No strong anomaly indicator")

    reasons.append(r)

df['risk_reasons'] = reasons

# ===============================
# 🔟 Save Report
# ===============================

final_output = df[['amount','risk_score','risk_grade','risk_reasons']]

final_output.to_csv("corruption_risk_report.csv", index=False)

print("✅ corruption_risk_report.csv created successfully!")



Original Shape: (1576, 7)
After cleaning: (1576, 7)
Feature Shape: (1576, 6)
✅ corruption_risk_report.csv created successfully!


In [9]:
print(df.columns)

Index(['amount', 'duration_days', 'num_bidders', 'proc_method', 'buyer',
       'year', 'status', 'buyer_frequency', 'proc_method_encoded',
       'status_encoded', 'anomaly_score', 'risk_score', 'risk_grade',
       'risk_reasons'],
      dtype='str')


In [2]:

import joblib

joblib.dump(model, "risk_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(risk_scaler, "risk_scaler.pkl")

print("Model and scaler saved!")

Model and scaler saved!


In [19]:
import pandas as pd
import joblib

# ==============================
# 🔹 Load Model and Scalers
# ==============================

model = joblib.load("risk_model.pkl")
scaler = joblib.load("scaler.pkl")
risk_scaler = joblib.load("risk_scaler.pkl")

# Feature list (MUST match training exactly)
features = [
    'amount',
    'duration_days',
    'num_bidders',
    'buyer_frequency',
    'proc_method_encoded',
    'status_encoded'
]

# ==============================
# 🔹 Manual Input Section
# ==============================

print("\n====== ENTER TENDER DETAILS ======")

amount = float(input("Enter Contract Amount: "))
duration_days = float(input("Enter Tender Duration (days): "))
num_bidders = float(input("Enter Number of Bidders: "))
buyer_frequency = float(input("Enter Buyer Frequency: "))
proc_method_encoded = float(input("Enter Procurement Method Code (0,1,2..): "))
status_encoded = float(input("Enter Status Code (0,1,2..): "))

# ==============================
# 🔹 Create DataFrame Input
# ==============================

manual_input = pd.DataFrame([{
    'amount': amount,
    'duration_days': duration_days,
    'num_bidders': num_bidders,
    'buyer_frequency': buyer_frequency,
    'proc_method_encoded': proc_method_encoded,
    'status_encoded': status_encoded
}])

# Ensure column order matches training
manual_input = manual_input[features]

# ==============================
# 🔹 Feature Scaling
# ==============================

manual_scaled = scaler.transform(manual_input)

# ==============================
# 🔹 Get Anomaly Score
# ==============================

anomaly_score = model.decision_function(manual_scaled)[0]

# ==============================
# 🔹 Convert to Risk Score (0–100)
# ==============================

# Use DataFrame to avoid sklearn warning
risk_input = pd.DataFrame(
    {'anomaly_score': [-anomaly_score]}
)

risk_score = risk_scaler.transform(risk_input)[0][0]

# ==============================
# 🔹 Risk Grade
# ==============================

if risk_score > 70:
    risk_grade = "High"
elif risk_score > 40:
    risk_grade = "Medium"
else:
    risk_grade = "Low"

# ==============================
# 🔹 Risk Reasons Logic
# ==============================

reasons = []

if num_bidders <= 1:
    reasons.append("Low bidder competition")

if duration_days < 3:
    reasons.append("Very short tender duration")

if amount > 10000000:  # Adjust based on dataset statistics
    reasons.append("Unusually high contract value")

if buyer_frequency > 50:
    reasons.append("High buyer concentration")

if not reasons:
    reasons.append("No strong anomaly indicator")

# ==============================
# 🔹 Final Output
# ==============================

print("\n====== RISK ANALYSIS RESULT ======")
print("Risk Score:", round(risk_score, 2))
print("Risk Grade:", risk_grade)
print("Reasons:")
for r in reasons:
    print("-", r)


====== ENTER TENDER DETAILS ======

====== RISK ANALYSIS RESULT ======
Risk Score: 67.74
Risk Grade: Medium
Reasons:
- Unusually high contract value
